# QTM 151 — Quiz 4 (Lecture 18) — Fully Solved Notebook

This notebook contains **complete, runnable solutions** to the six quiz questions based on **Lecture 18** topics:
- Pandas `DataFrame.plot()` vs `matplotlib.pyplot` usage
- Building and plotting **quadratic models**
- Date parsing and **time series** line plots
- **Normalization** to non‑dimensional % growth
- **Boolean masks** and highlighting with `plt.fill_between`
- **Finite differences** (`.diff`) to approximate derivatives

> **How to use**: If you have the provided CSVs, set the file paths in the **Config** cell below.  
> If files are missing, the notebook will **auto‑generate small demo datasets** so you can still run every solution end‑to‑end.


In [ ]:
# === Config (edit paths if you have the real files) ===

# Problem 2: Delhi weather (2017) CSV path
DELHI_WEATHER_CSV = "/mnt/data/delhi_weather_2017.csv"  # update if needed

# Problem 3-5: FX conversion rates CSV path
FX_RATES_CSV = "/mnt/data/fx_rates.csv"  # update if needed

# Threshold for Problems 4-5
EUR_THRESHOLD = 1.20  # example threshold for EUR->USD

# General plotting imports (Lecture 18 material)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Guarantee reasonably sized plots in notebook outputs
plt.rcParams['figure.figsize'] = (8, 5)


In [ ]:
# === Helpers: synthesize datasets if files are not found ===

from pathlib import Path

def ensure_delhi_weather_csv(path):
    p = Path(path)
    if p.exists():
        return str(p)
    # Make a synthetic daily 2017 time series with plausible temps and wind
    rng = pd.date_range("2017-01-01", "2017-12-31", freq="D")
    # Simple seasonal temp (C) and wind (m/s) pattern
    days = np.arange(len(rng))
    temp = 18 + 12*np.sin(2*np.pi*days/365 - np.pi/2) + np.random.normal(scale=1.5, size=len(rng))
    wind = 2 + 1.2*np.sin(2*np.pi*days/30) + np.random.normal(scale=0.4, size=len(rng))
    df = pd.DataFrame({"date": rng, "temperature_C": temp, "wind_speed_mps": wind})
    df.to_csv(p, index=False)
    return str(p)

def ensure_fx_rates_csv(path):
    p = Path(path)
    if p.exists():
        return str(p)
    # Synthetic FX rates (daily): EURUSD and GBPUSD with gentle drift & noise
    rng = pd.date_range("2020-01-01", "2020-12-31", freq="D")
    n = len(rng)
    # Start values
    eur0, gbp0 = 1.10, 1.30
    # Random walk with small drift
    eur = eur0 + np.cumsum(np.random.normal(0, 0.001, size=n)) + 0.0002*np.arange(n)
    gbp = gbp0 + np.cumsum(np.random.normal(0, 0.0012, size=n)) - 0.0001*np.arange(n)
    df = pd.DataFrame({"date": rng, "EURUSD": eur, "GBPUSD": gbp})
    df.to_csv(p, index=False)
    return str(p)


## Problem 1 — Fixing a Wrong Plot for a Quadratic Model

**Setup**: Assume the previous (given) cell correctly computed quadratic coefficients `a, b, c` for a model  
\begin{equation}
\hat{y} = a x^2 + b x + c.
\end{equation}

**Common plotting mistakes to fix (Lecture 18):**
- Plotting predictions against the **index** rather than the **matching x** values.
- Not **sorting** `x` before drawing a continuous prediction curve (causes a zig‑zag line).
- Mixing up variables in `plt.plot(x, y_pred)` (axes flipped).

**Goal**: Generate a correct plot: scatter of `(x, y)` and a **smooth model line** of `(x_sorted, yhat_sorted)`.


In [ ]:
# --- Simulate the "given" correct coefficients and data ---
rng = np.random.default_rng(42)
x = np.linspace(-3, 3, 40)
true_a, true_b, true_c = 0.7, -0.6, 2.0
y = true_a*x**2 + true_b*x + true_c + rng.normal(0, 0.5, size=x.size)

# "Given" model coefficients (assume computed correctly elsewhere)
a, b, c = np.polyfit(x, y, 2)

# --- Correct plotting ---
# 1) Scatter the original data
plt.figure()
plt.scatter(x, y, label="data")

# 2) Sort x to draw a smooth model curve left-to-right
order = np.argsort(x)
x_sorted = x[order]
yhat_sorted = a*x_sorted**2 + b*x_sorted + c

# 3) Plot the model line against the sorted x
plt.plot(x_sorted, yhat_sorted, label="quadratic fit")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Quadratic Model: data vs. fitted curve")
plt.legend()
plt.show()


## Problem 2 — Delhi 2017 Weather: Pandas `.plot()` Line Plot

**Task**: Read a CSV of Delhi (2017) weather with a time column and numeric columns for **temperature** and **wind speed**.  
Using **`DataFrame.plot()`** (not `plt.plot`), produce a **single line figure** with:
- time vs. temperature
- time vs. wind speed

**Lecture 18 tips**:
- Parse the date column and set it as the **index** for nicer time‑series plots.
- Use `df[['colA','colB']].plot()` to plot multiple series on one figure.


In [ ]:
# Ensure we have a CSV to read (creates a demo one if missing)
delhi_csv = ensure_delhi_weather_csv(DELHI_WEATHER_CSV)

# Read and parse
df_delhi = pd.read_csv(delhi_csv, parse_dates=["date"])
df_delhi = df_delhi.set_index("date").sort_index()

# Plot using the Pandas DataFrame .plot() method (single axes, two lines)
ax = df_delhi[["temperature_C", "wind_speed_mps"]].plot()
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.set_title("Delhi (2017): Temperature and Wind Speed")
plt.show()

# Peek at the head for verification
df_delhi.head()


## Problem 3 — FX: Non‑Dimensional % Growth and Plot

**Task**: CSV contains **EURUSD** and **GBPUSD** over time. Compute **non‑dimensional percentage growth** for each series and plot them together.

**Lecture 18 pattern** (normalize to first value):
\begin{equation}
\text{%Growth}(t) = \left(\frac{x_t}{x_{t_0}} - 1\right) \times 100.
\end{equation}


In [ ]:
# Ensure FX CSV exists
fx_csv = ensure_fx_rates_csv(FX_RATES_CSV)

# Read and normalize
df_fx = pd.read_csv(fx_csv, parse_dates=["date"]).sort_values("date").set_index("date")

for col in ["EURUSD", "GBPUSD"]:
    base = df_fx[col].iloc[0]
    df_fx[col + "_pct_growth"] = (df_fx[col] / base - 1.0) * 100.0

# Plot both percentage growth series together (either Pandas plot or plt)
ax = df_fx[["EURUSD_pct_growth", "GBPUSD_pct_growth"]].plot()
ax.set_xlabel("date")
ax.set_ylabel("% growth from first day")
ax.set_title("EURUSD vs GBPUSD — Non‑Dimensional % Growth")
plt.show()

df_fx.head()


## Problem 4 — Boolean Column for Threshold Check

**Task**: In the same FX DataFrame, create a **Boolean column** that checks whether **EURUSD** exceeds a threshold (e.g., `EUR_THRESHOLD`).

**Lecture 18**: Boolean masks are simple comparisons on columns: `df["flag"] = df["col"] > value`.


In [ ]:
df_fx["EUR_above_threshold"] = df_fx["EURUSD"] > EUR_THRESHOLD
df_fx[["EURUSD", "EUR_above_threshold"]].head(10)


## Problem 5 — Highlight Regions with `plt.fill_between`

**Task**: Plot the **EURUSD** line and **highlight** the regions where it is above the threshold using `plt.fill_between` with the Boolean mask from Problem 4.

**Lecture 18**:
- Use `mask = df["EURUSD"] > threshold`
- Then `plt.fill_between(x, y, where=mask, alpha=...)` to shade the regions.


In [ ]:
# Prepare series
x = df_fx.index.values
y = df_fx["EURUSD"].values
mask = df_fx["EUR_above_threshold"].values

plt.figure()
plt.plot(df_fx.index, df_fx["EURUSD"], label="EURUSD")
plt.fill_between(df_fx.index, df_fx["EURUSD"], where=mask, alpha=0.25, label=f"EURUSD > {EUR_THRESHOLD}")
plt.xlabel("date")
plt.ylabel("EURUSD")
plt.title("EURUSD with Threshold Highlight")
plt.legend()
plt.show()


## Problem 6 — Finite Differences `.diff()` and Derivative Check

**Task**:
1. Create a DataFrame with columns:
   - `time`: evenly spaced points between \(0\) and \(4\pi\)
   - `sin`: \(\sin(t)\)
   - `cos`: \(\cos(t)\)

2. Use `.diff()` to compute a discrete derivative:
\begin{equation}
\text{d\_sin\_dt}_i \approx \frac{\sin(t_i) - \sin(t_{i-1})}{t_i - t_{i-1}}.
\end{equation}

3. Plot `(time, cos)` and `(time, d_sin_dt)` together to show they are close, as calculus tells us \(\frac{d}{dt}\sin t = \cos t\).

**Lecture 18 notes**:
- Use `.diff()` on both numerator and denominator, then divide.
- The first element will be `NaN` — OK for plotting.


In [ ]:
# Build the time grid and functions
N = 400
t0, t1 = 0.0, 4.0*np.pi
time = np.linspace(t0, t1, N)
sin_t = np.sin(time)
cos_t = np.cos(time)

df_trig = pd.DataFrame({"time": time, "sin": sin_t, "cos": cos_t})

# Finite difference derivative: d(sin)/dt ≈ diff(sin) / diff(time)
df_trig["d_sin_dt"] = df_trig["sin"].diff() / df_trig["time"].diff()

# Plot cos and the finite-difference derivative together
plt.figure()
plt.plot(df_trig["time"], df_trig["cos"], label="cos(t)")
plt.plot(df_trig["time"], df_trig["d_sin_dt"], label="d(sin)/dt (finite diff)")
plt.xlabel("time")
plt.ylabel("value")
plt.title("cos(t) vs. finite-difference derivative of sin(t)")
plt.legend()
plt.show()

df_trig.head()
